In [1]:
import numpy as np
import pandas as pd
import requests
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import datetime
from pathlib import Path 
import re

Need for this file:

- The imports above
- locations data: Path to a locations.csv file (to merge location name to location code). Path will be read in as a pd.DataFrame
- current FluSMH GitHub repository local clone. The file will use the abs path to the `model-output` directory:
    e.g. "your/path/to/flu-scenario-modeling-hub/model-output"
- archive FluSMH GitHub repository locial clone. The file will use the abs path to the `data-processed` directory
    e.g. "your/path/to/flu-scenario-modeling-hub_archive/data-processed"
- output path (abs) for processed files
    replace in saving for loop

In [2]:
# modified from respi

LOCATIONS_ABBREV = [
        'AL', 'AK', 'AZ', 'AR', 'CA', 'CO', 'CT', 'DE', 'DC', 'FL', 'GA', 
        'HI', 'ID', 'IL', 'IN', 'IA', 'KS', 'KY', 'LA', 'ME', 'MD', 'MA', 
        'MI', 'MN', 'MS', 'MO', 'MT', 'NE', 'NV', 'NH', 'NJ', 'NM', 'NY', 
        'NC', 'ND', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC', 'SD', 'TN', 'TX', 
        'UT', 'VT', 'VA', 'WA', 'WV', 'WI', 'WY', 'US'
    ]

class NHSNDataProcessor:
    def __init__(self, resource_id, replace_column_names: bool = True):
        self.replace_column_names = replace_column_names
        self.data_url = "https://data.cdc.gov/resource/" + f"{resource_id}.json"
        self.metadata_url = "https://data.cdc.gov/api/views/" + f"{resource_id}.json"
        self.output_dict = {}

        self._process_data()

    
    def _process_data(self):
        """Fetches, processes, and structures NHSN data into self.output_dict"""
        # Get data set up 
        data = pd.DataFrame(self._retrieve_data_from_endpoint_aslist()) # read from endpoint
        data = data.drop(columns=['respseason'])
        non_numeric_cols = ['jurisdiction', 'weekendingdate'] # make numeric cols not strings
        for col in data.columns:
            if col not in non_numeric_cols:
                data[col] = pd.to_numeric(data[col], errors='raise')
        data = data.replace(np.nan, value=None) # cleanse NaN values 
        data.loc[data['jurisdiction'].str.lower() == 'usa', 'jurisdiction'] = 'US' # change USA jurisdiction to US
        data = data[data['jurisdiction'].isin(LOCATIONS_ABBREV)].copy() # filter out unwanted regions
        data['weekendingdate'] = pd.to_datetime(data['weekendingdate']).dt.strftime('%Y-%m-%d') # ensure date columns are dates
        # Get metadata set up
        cdc_metadata = (requests.get(self.metadata_url)).json() 

        if self.replace_column_names: 
            data = self._replace_column_names(data, cdc_metadata) 
            data = data.sort_values(by=['Geographic aggregation', 'Week Ending Date'])
            self.output_dict["processed_df"] = data

    def _retrieve_data_from_endpoint_aslist(self) -> list[dict]:
        """Downloads NHSN data from the endpoint with pagination and retries."""
        
        session = requests.Session()
        retries = Retry(total=5,
                        backoff_factor=1,
                        status_forcelist=[500, 502, 503, 504])
        session.mount('https://', HTTPAdapter(max_retries=retries))
        
        all_data = []
        offset = 0
        batch_size = 1000
        while True:
            params = {"$limit": batch_size, "$offset": offset}
            try:
                # Use the configured session to make the request
                data_response = session.get(self.data_url, params=params, timeout=30)
                data_response.raise_for_status()
                batch_data = data_response.json()
                if not batch_data:
                    break
                all_data.extend(batch_data)
                offset += batch_size
                time.sleep(0.1)
            except Exception as e:
                raise
        return all_data
    
    def _replace_column_names(self, data: pd.DataFrame, cdc_metadata: dict) -> pd.DataFrame:
        """Replace short-form column names with long-form column names"""
        column_name_map = {
                col_info['fieldName']: col_info['name']
                for col_info in cdc_metadata['columns']
            }
        return data.rename(columns=column_name_map, errors="ignore")


NHSNData = NHSNDataProcessor(resource_id='ua7e-t2fy')

    

In [3]:
data = NHSNData.output_dict['processed_df'] # only has 2024-25 seasons for the metrics we need
# has all weeks for all location in 2024 and 2025 so far though! 

# initial column filtering
data = data[[ 
    "Week Ending Date", 
    "Geographic aggregation",
    "Number of Pediatric Influenza Admissions, 0-4 years",
    "Number of Pediatric Influenza Admissions, 5-17 years",
    "Total Pediatric Influenza Admissions",
    "Number of Adult Influenza Admissions, 18-49 years",
    "Number of Adult Influenza Admissions, 50-64 years",
    "Number of Adult Influenza Admissions, 65-74 years",
    "Number of Adult Influenza Admissions, 75+ years",
    "Total Adult Influenza Admissions",
    ]].copy()

# renaming, removing, and small calculations
data = data.fillna(0) # fill all Na values w/ 0
data['0-130'] = data["Total Pediatric Influenza Admissions"] + data["Total Adult Influenza Admissions"]
data['65+'] = data['Number of Adult Influenza Admissions, 65-74 years'] + data['Number of Adult Influenza Admissions, 75+ years']
data = data.rename(columns={
    'Week Ending Date': 'week_enddate', 
    'Number of Pediatric Influenza Admissions, 0-4 years': '0-4',
    'Number of Pediatric Influenza Admissions, 5-17 years': '5-17',
    'Number of Adult Influenza Admissions, 18-49 years': '18-49',
    'Number of Adult Influenza Admissions, 50-64 years': '50-64',
    })
data = data.drop(columns=['Number of Adult Influenza Admissions, 65-74 years', 'Number of Adult Influenza Admissions, 75+ years', 'Total Pediatric Influenza Admissions', 'Total Adult Influenza Admissions'])

# remove US loc
# data = data[data['Geographic aggregation'] != 'US'], opting to keep US in now
# cast week_enddate column as date
data['week_enddate'] = pd.to_datetime(data['week_enddate'])
# add fluseason column
data['fluseason'] = data['week_enddate'].dt.year
data.loc[data['week_enddate'].dt.month < 8, 'fluseason'] -= 1
# keep only 2024 fluseason (that's where the age group we need start, and we won't use current season)
data = data[data['fluseason'].isin([2024])]
# add location_code, sample, datasetH1, datasetH2 columns
locations = pd.read_csv('/Users/emprzy/Documents/work/miscellaneous/locations.csv')
data = data.merge(
    locations[['abbreviation', 'location']],
    left_on='Geographic aggregation',
    right_on='abbreviation',
    how='left'
)
data = data.rename(columns={'location': 'location_code'}).drop(columns=['abbreviation', 'Geographic aggregation'])
data['sample'] = "1"
data['datasetH1'] = 'NHSN'
data['datasetH2'] = 'NHSN'
# add season_week column
data = data.sort_values(['location_code', 'fluseason', 'week_enddate'])
data['season_week'] = data.groupby(['location_code', 'fluseason']).cumcount() + 1
# add fluseason_fraction
def get_season_fraction(ts, start_month: int, start_day: int): # modified from influpaint season_axis.py
    if pd.isna(ts):
        return float('nan')
    if isinstance(ts, datetime.datetime):
        ts = ts.date()
    try:
        season_start = datetime.date(ts.year, start_month, start_day)
    except AttributeError:
        season_start = datetime.date(ts.year, start_month, start_day)
    if ts < season_start:
        season_start = datetime.date(ts.year - 1, start_month, start_day)
    
    days_since_start = (ts - season_start).days
    return days_since_start / 365

data['fluseason_fraction'] = data['week_enddate'].apply(
    get_season_fraction, 
    start_month=10, 
    start_day=1
)

NHSN = data

# melt NHSN age grouping columns into singular `age_group`
NHSN = NHSN.rename(columns={'65+': '65-130'})
age_columns = ['0-4', '5-17', '18-49', '50-64', '0-130', '65-130']
id_vars = [col for col in NHSN.columns if col not in age_columns]
NHSN = NHSN.melt(
    id_vars=id_vars,
    value_vars=age_columns,
    var_name='age_group',
    value_name='value'
)

/var/folders/pf/s416pvp93gd610_55fzf_f280000gp/T/ipykernel_53075/2467725948.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.fillna(0) # fill all Na values w/ 0


In [4]:
LOCATIONS = set(data['location_code'])

In [5]:


def fetch_SMH_submissions(base_path: str) -> list:
    base_path = Path(base_path)
    keep_files = []
    YEAR_PATTERN = re.compile(r'^(2022-08|2023|2024|2025)')
    # YEAR_PATTERN = re.compile(r'^(2023|2024|2025)')

    for folder in base_path.iterdir(): # if it's a folder
        if folder.is_dir():
            
            # iterate over files in that folder
            for file in folder.iterdir():
                if file.is_file() and not file.name.startswith('.'):
                    full_file_path = file.resolve()

                    if full_file_path.suffix.lower() in ['.csv', '.parquet']:
                        if YEAR_PATTERN.match(file.name):
                            keep_files.append(full_file_path)

                    else: continue # skip files that aren't csv or parquet

                else: continue # skip folders once inside of each models' dir

        else: # if it's a file (skipping .md files)
            continue

    return keep_files


def parse_SMH_submissions(file_paths: dict[str: Path]) -> dict[str: Path]:
    # return object
    good_files = {}
    AGE_GROUPS = {'0-130', '0-4', '18-49', '5-17', '50-64', '65-130'}

    for _, list_of_paths in file_paths.items():
        for path in list_of_paths:
            if path.suffix.lower() == '.csv':
                data = pd.read_csv(path)
            elif path.suffix.lower() == '.parquet':
                data = pd.read_parquet(path)
            else:
                raise ValueError(f'Unexpected file type received: {path.suffix}')

            if not 'output_type' in data.columns: # 2022 round 1 doesn't have output_type column
                if 'type' in data.columns:
                    data = data[data['type'] == 'point'] # only keep sample output type
            else:
                data = data[data['output_type'] == 'sample'] # only keep sample output_type

            # check if all required age groups are present
            if not (AGE_GROUPS.issubset(set(data['age_group']))):
                continue
            
            # check by age group
            is_file_valid = True
            age_group_gbo = data.groupby('age_group')
            for name, age_group_df in age_group_gbo:
                if name not in AGE_GROUPS:
                    continue
                has_locations = len(age_group_df['location'].unique()) >= 20
                has_target = age_group_df['target'].str.contains('inc hosp', case=False, na=False).any()
                if not (has_locations and has_target):
                    is_file_valid = False
                    break 

            if is_file_valid:
                good_files[path.name] = {
                    'file_path': path,
                    'scenarios': set(data['scenario_id']),
                    'num_locations': len(data['location'].unique())
                }

    return good_files

In [6]:
base_paths = {"past": "/Users/emprzy/Documents/work/flu-scenario-modeling-hub_archive/data-processed", "current": "/Users/emprzy/Documents/work/flu-scenario-modeling-hub/model-output"}
files_to_parse = {}
for origin, path in base_paths.items():
    files_to_parse[origin] = fetch_SMH_submissions(base_path=path)

good_files = parse_SMH_submissions(file_paths=files_to_parse) # note that there are none from 2022! 15 total good files

/var/folders/pf/s416pvp93gd610_55fzf_f280000gp/T/ipykernel_53075/3944370396.py:37: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(path)
/var/folders/pf/s416pvp93gd610_55fzf_f280000gp/T/ipykernel_53075/3944370396.py:37: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(path)


In [7]:
def impute_season_week(data: pd.DataFrame, fluseason: int, trajectories: int) -> pd.DataFrame: # critical: assumes only one origin_date per df
    if len(set(data['origin_date'])) != 1:
        raise ValueError(f"Len of unique origin_dates is {len(set(data['origin_date']))}") 

    # prelim set up
    zero_horizon = pd.to_datetime(data['origin_date'].iloc[0])
    one_horizon = pd.to_datetime(data['origin_date'].iloc[0])+pd.Timedelta(days=7)
    horizon_to_week_enddate = {0: {"week_enddate": zero_horizon}, 1: {"week_enddate": one_horizon}}

    # add all necessary weeks to dict
    desired_start = pd.to_datetime(datetime.datetime(fluseason, 8, 1))
    current_date = zero_horizon
    week_offset = 0
    while current_date > desired_start:
        current_date -= pd.Timedelta(days=7) 
        week_offset += 1
    for i in range(1,week_offset):
        j = i*-1
        horizon_to_week_enddate[j] = {}
        horizon_to_week_enddate[j]["week_enddate"] = (horizon_to_week_enddate[0]["week_enddate"] - pd.Timedelta(days=7*i))
    for i in range(2,54):
        horizon_to_week_enddate[i] = {}
        horizon_to_week_enddate[i]["week_enddate"] = (horizon_to_week_enddate[0]["week_enddate"] + pd.Timedelta(days=7*i))

    # trim off horizons we don't need
    next_season = fluseason + 1
    keys_to_delete = []
    for key, value in horizon_to_week_enddate.items():
        if value["week_enddate"] > pd.to_datetime(datetime.datetime(next_season, 8, 1)):
            keys_to_delete.append(key)
    for key in keys_to_delete:
        del horizon_to_week_enddate[key]

    # add season_week to dict
    sorted_keys = sorted(horizon_to_week_enddate.keys())
    for i, key in enumerate(sorted_keys, start=1):
        horizon_to_week_enddate[key]["season_week"] = i 

    # add missing horizons
    missing_horizons = set(horizon_to_week_enddate.keys()) - set(data['horizon'])
    if missing_horizons:
        unique_groups = data[['scenario_id', 'location', 'age_group']].drop_duplicates()
        new_rows_list = []
        for h in missing_horizons:
            temp_df = unique_groups.copy()
            temp_df['horizon'] = h
            temp_df['value'] = 0
            temp_df = temp_df.loc[temp_df.index.repeat(trajectories)].reset_index(drop=True) 
            new_rows_list.append(temp_df)
        missing_data_df = pd.concat(new_rows_list, ignore_index=True)
        data = pd.concat([data, missing_data_df], ignore_index=True)
    
    
    # match horizon to season_week, week_enddate
    sw_map = {k: v['season_week'] for k, v in horizon_to_week_enddate.items()}
    we_map = {k: v['week_enddate'] for k, v in horizon_to_week_enddate.items()}
    data['season_week'] = data['horizon'].map(sw_map)
    data['week_enddate'] = data['horizon'].map(we_map)

    # add fluseason_fraction column (function used is defined above; with NHSN data)
    data['fluseason_fraction'] = data['week_enddate'].apply(
        get_season_fraction, 
        start_month=10, 
        start_day=1
    )

    # add sample column (CRITICAL: assumes all trajectory numbers are equal across all combinations)
    sort_cols = ['scenario_id', 'location', 'age_group', 'season_week']
    data = data.sort_values(by=sort_cols).reset_index(drop=True)
    data['sample'] = np.tile(np.arange(1, trajectories + 1), len(data) // trajectories)

    # drop useless columns
    cols_to_drop = [c for c in ['origin_date', 'target', 'output_type', 'output_type_id', 'horizon'] if c in data.columns]
    data = data.drop(columns=cols_to_drop)
    data = data.rename(columns={'location': 'location_code'})

    return data


def filter_smh(data: pd.DataFrame, fluseason: int, datasetH1: str, datasetH2: str, trajectories: int) -> pd.DataFrame:
    data = data[data['output_type'] == 'sample'] # only keep sample output type
    data = data[data['target'] == 'inc hosp'] # only keep inc hosp target
    data['location'] = data['location'].astype(str) # ensure location column is type <'str'>
    data = data[data['location'].isin(LOCATIONS)] # only keep 50 states + D.C. + U.S. (52 total locations)
    data = impute_season_week(data=data, fluseason=fluseason, trajectories=trajectories) # add season_week, week_enddate
    data['fluseason'] = fluseason # add fluseason column based on explicit specification
    # add identifying columns
    data['datasetH1'] = datasetH1
    datasetH2_list = []
    for row in data.itertuples():
        value = datasetH2 + row.scenario_id
        datasetH2_list.append(value)
    data['datasetH2'] = datasetH2_list
    data = data.drop(columns=['scenario_id'])
    return data


def get_trajectory_count(data: pd.DataFrame) -> int:
    """
    Helper function to count how many trajectories per model, per location, per SMH scenario.

    Arbitrary grouping; just needed a number.
    """
    x = data[
        (data['scenario_id'] == data['scenario_id'].iloc[0]) & 
        (data['horizon'] == 1) & 
        (data['location'] == "37") & 
        (data['age_group'] == '0-4') & 
        (data['target'] == 'inc hosp') & 
        (data['output_type'] == 'sample')
    ]
    return len(x)
    



In [8]:
# trajectories counted using get_trajectory_count(), but code for this is ommitted from this file

one = pd.read_parquet("/Users/emprzy/Documents/work/flu-scenario-modeling-hub_archive/data-processed/USC-SIkJalpha/2023-09-03-USC-SIkJalpha.parquet")
one = filter_smh(one, 2023, datasetH1="SMH_R4", datasetH2="round4_USC-SIkJalpha_", trajectories=100)

two = pd.read_parquet("/Users/emprzy/Documents/work/flu-scenario-modeling-hub_archive/data-processed/USC-SIkJalpha/2024-08-11-USC-SIkJalpha.parquet")
two = filter_smh(two, 2024, datasetH1="SMH_R5", datasetH2="round5_USC-SIkJalpha_", trajectories=100)
two = two.drop(columns=['run_grouping', 'stochastic_run']) 

three = pd.read_parquet("/Users/emprzy/Documents/work/flu-scenario-modeling-hub_archive/data-processed/NotreDame-FRED/2024-08-11-NotreDame-FRED.gz.parquet")
three = filter_smh(three, 2024, datasetH1="SMH_R5", datasetH2="round5_NotreDame-FRED_", trajectories=200)
three = three.drop(columns=['run_grouping', 'stochastic_run']) 

four = pd.read_parquet("/Users/emprzy/Documents/work/flu-scenario-modeling-hub_archive/data-processed/SigSci-SWIFT/2024-08-11-SigSci-SWIFT.parquet")
four = filter_smh(four, 2024, datasetH1="SMH_R5", datasetH2="round5_SigSci-SWIFT_", trajectories=300)
four = four.drop(columns=['run_grouping', 'stochastic_run'])

five = pd.read_parquet("/Users/emprzy/Documents/work/flu-scenario-modeling-hub_archive/data-processed/UVA-FluXSim/2024-08-11-UVA-FluXSim.parquet")
five = filter_smh(five, 2024, datasetH1="SMH_R5", datasetH2="round5_UVA-FluXSim_", trajectories=100)
five = five.drop(columns=['run_grouping', 'stochastic_run'])

six = pd.read_parquet("/Users/emprzy/Documents/work/flu-scenario-modeling-hub_archive/data-processed/UVA-EscapeFlu/2024-08-11-UVA-EscapeFlu.parquet")
six = filter_smh(six, 2024, datasetH1="SMH_R5", datasetH2="round5_UVA-EscapeFlu_", trajectories=120)
six = six.drop(columns=['run_grouping', 'stochastic_run'])

seven = pd.read_parquet("/Users/emprzy/Documents/work/flu-scenario-modeling-hub_archive/data-processed/ACCIDDA-FlepiMoP/2024-08-11-ACCIDDA-FlepiMoP.gz.parquet")
seven = filter_smh(seven, 2024, datasetH1="SMH_R5", datasetH2="round5_ACCIDDA-FlepiMoP_", trajectories=100)
seven = seven.drop(columns=['run_grouping', 'stochastic_run'])

eight = pd.read_parquet("/Users/emprzy/Documents/work/flu-scenario-modeling-hub_archive/data-processed/MOBS_NEU-GLEAM_FLU/2024-08-11-MOBS_NEU-GLEAM_FLU.gz.parquet")
eight = filter_smh(eight, 2024, datasetH1="SMH_R5", datasetH2="round5_MOBS_NEU-GLEAM_FLU_", trajectories=300)
eight = eight.drop(columns=['run_grouping', 'stochastic_run'])

# !!! THE FOLLOWING DATASETS REPRESENT THE CURRENT SMH ROUND (2025-26 season) !!! 

nine = pd.read_parquet("/Users/emprzy/Documents/work/flu-scenario-modeling-hub/model-output/UT-ImmunoSEIRS/2025-08-10-UT-ImmunoSEIRS.gz.parquet")
nine = filter_smh(nine, 2025, datasetH1="SMH_R6", datasetH2="round6_UT-ImmunoSEIRS_", trajectories=300)
nine = nine.drop(columns=['run_grouping', 'stochastic_run'])

ten = pd.read_parquet("/Users/emprzy/Documents/work/flu-scenario-modeling-hub/model-output/CEPH-MetaFlu/2025-08-10-CEPH-MetaFlu.gz.parquet")
ten = filter_smh(ten, 2025, datasetH1="SMH_R6", datasetH2="round6_CEPH-MetaFlu_", trajectories=300)
ten = ten.drop(columns=['run_grouping', 'stochastic_run'])

eleven = pd.read_parquet("/Users/emprzy/Documents/work/flu-scenario-modeling-hub/model-output/UVA-FluXSim/2025-08-10-UVA-FluXSim.parquet")
eleven = filter_smh(eleven, 2025, datasetH1="SMH_R6", datasetH2="round6_UVA-FluXSim_", trajectories=300)
eleven = eleven.drop(columns=['run_grouping', 'stochastic_run'])

twelve = pd.read_parquet("/Users/emprzy/Documents/work/flu-scenario-modeling-hub/model-output/ACCIDDA-FlepiMoP/2025-08-10-ACCIDDA-FlepiMoP.gz.parquet")
twelve = filter_smh(twelve, 2025, datasetH1="SMH_R6", datasetH2="round6_ACCIDDA-FlepiMoP_", trajectories=600)
twelve = twelve.drop(columns=['run_grouping', 'stochastic_run'])

thirteen = pd.read_parquet("/Users/emprzy/Documents/work/flu-scenario-modeling-hub/model-output/MOBS_NEU-GLEAM_FLU/2025-08-10-MOBS_NEU-GLEAM_FLU.gz.parquet")
thirteen = filter_smh(thirteen, 2025, datasetH1="SMH_R6", datasetH2="round6_MOBS_NEU-GLEAM_FLU_", trajectories=600)
thirteen = thirteen.drop(columns=['run_grouping', 'stochastic_run'])

fourteen = pd.read_parquet("/Users/emprzy/Documents/work/flu-scenario-modeling-hub/model-output/UNCC-Hierbin/2025-08-10-UNCC-Hierbin.gz.parquet")
fourteen = filter_smh(fourteen, 2025, datasetH1="SMH_R6", datasetH2="round6_UNCC-Hierbin_", trajectories=300)
fourteen = fourteen.drop(columns=['run_grouping', 'stochastic_run'])

fifteen = pd.read_parquet("/Users/emprzy/Documents/work/flu-scenario-modeling-hub/model-output/PSI-M3/2025-08-10-PSI-M3.gz.parquet")
fifteen = filter_smh(fifteen, 2025, datasetH1="SMH_R6", datasetH2="round6_PSI-M3_", trajectories=600)
fifteen = fifteen.drop(columns=['run_grouping', 'stochastic_run'])

In [9]:
# files not separated by age group. save all data to CSV
finished_files = {
    "one": {"df": one, "name": "round4_USC-SIkJalpha.csv"}, 
    "two": {"df": two, "name": "round5_USC-SIkJalpha.csv"}, 
    "three": {"df": three, "name": "round5_NotreDame-FRED.csv"}, 
    "four": {"df": four, "name": "round5_SigSci-SWIFT.csv"}, 
    "five": {"df": five, "name": "round5_UVA-FluXSim.csv"},
    "six": {"df": six, "name": "round5_UVA-EscapeFlu.csv"}, 
    "seven": {"df": seven, "name": "round5_ACCIDDA-FlepiMoP.csv"}, 
    "eight": {"df": eight, "name": "round5_MOBS_NEU-GLEAM_FLU.csv"}, 
    "nine": {"df": nine, "name": "round6_UT-ImmunoSEIRS.csv"}, # start of current season (nine-fiften are un-saved to CSV)
    "ten": {"df": ten, "name": "round6_CEPH-MetaFlu.csv"}, 
    "eleven": {"df": eleven, "name": "round6_UVA-FluXSim.csv"}, 
    "twelve": {"df": twelve, "name": "round6_ACCIDDA-FlepiMoP.csv"}, 
    "thirteen": {"df": thirteen, "name": "round6_MOBS_NEU-GLEAM_FLU.csv"}, 
    "fourteen": {"df": fourteen, "name": "round6_UNCC-Hierbin.csv"}, 
    "fifteen": {"df": fifteen, "name": "round6_PSI-M3.csv"},
    "NHSN": {"df": NHSN, "name": "2024_NHSN_surveillance.csv"} # surveillance
}
for file, info in finished_files.items():
    full_path = '/Users/emprzy/Documents/work/miscellaneous/influpaint_data' + '/' + info["name"]
    info["df"].to_csv(full_path, index=False)

training_data = pd.concat([one, two, three, four, five, six, seven, eight, NHSN]) # includes non-current SMH forecasts and surveillance data
all_data = pd.concat([
    one, two, three, four, five, six, seven,
    eight, nine, ten, eleven, twelve,
    thirteen, fourteen, fifteen, NHSN
])
training_data.to_csv('/Users/emprzy/Documents/work/miscellaneous/influpaint_data/training_data.csv', index=False)
all_data.to_csv('/Users/emprzy/Documents/work/miscellaneous/influpaint_data/all_data.csv', index=False)

Datatypes for each column:
- week_enddate: pandas._libs.tslibs.timestamps.Timestamp
- location_code: str
- value: numpy.float64
- fluseason_fraction: numpy.float64
- season_week: numpy.int64
- fluseason: numpy.int64
- datasetH1: str
- datasetH2: str
- sample: str

# --- DATA IS ALIGNED WITH THE GOAL AT THIS POINT! ---

In [ ]:
t = pd.read_csv(
    "/Users/emprzy/Documents/work/miscellaneous/influpaint_data/training_data.csv",
    dtype={
        'location_code': str,
        'value': np.float64,
        'fluseason_fraction': np.float64,
        'season_week': np.int64,
        'fluseason': np.int64,
        'datasetH1': str,
        'datasetH2': str,
        'sample': str  
    },
    parse_dates=['week_enddate']
)

In [ ]:
for dH1 in t['datasetH1'].unique():
    h1df= t[t['datasetH1'] == dH1]
    print(f"datasetH1: {dH1}, nH2= {len(h1df['datasetH2'].unique())}")
    for dH2 in h1df['datasetH2'].unique():
        h2df = h1df[h1df['datasetH2'] == dH2]
        print(f" -  datasetH2: {dH2}, shape: {h2df.shape}, years: {len(h2df['fluseason'].unique())}, samples: {len(h2df['sample'].unique())} ===> n_frames={len(h2df['fluseason'].unique())* len(h2df['sample'].unique())}")


In [14]:
goal = pd.read_parquet("/Users/emprzy/Documents/work/miscellaneous/josephs_processed_influpaint_data.parquet")
goal.head(6)

,week_enddate,location_code,value,fluseason_fraction,season_week,fluseason,datasetH1,datasetH2,sample
0,2010-10-09,02,0.875146,0.189041,10,2010,fluview,fluview,1
1,2010-10-16,02,1.128270,0.208219,11,2010,fluview,fluview,1
2,2010-10-23,02,0.586042,0.227397,12,2010,fluview,fluview,1
3,2010-10-30,02,0.967742,0.246575,13,2010,fluview,fluview,1
4,2010-11-06,02,0.683851,0.265753,14,2010,fluview,fluview,1
5,2010-11-13,02,0.951904,0.284932,15,2010,fluview,fluview,1


In [ ]:
# helper function i created to check how many data types are in a certain column
def type_count(df: pd.DataFrame, col_name: str) -> dict[str, int]:
    types = {}
    for value in df[col_name]:
        current_type = str(type(value))
        if current_type not in types:
            types[current_type] = 1
        else:
            types[current_type] += 1
            
    return types